# TransferGuard — REFINE + Early Detection (CIFAR-10)

**New standalone notebook · Local Windows / RTX 2050 · no source retraining**

Research reference: [Xu et al., REFINE, ICLR 2026](https://github.com/Xyc-arch/ReFine-residual-feature-integration). This notebook implements the paper's *EnhancedCNN concatenation* approach using the **40% and 80% source BigCNN checkpoints you already trained**.

We will:
1. Load the existing 40%-noise and 80%-noise source models (not retrain them).
2. Train the REFINE target model separately for each source (30 epochs each).
3. Compare against previously saved scratch and frozen-transfer histories.
4. Explore a simple **TransferGuard prototype**: feature separability + first-five-epoch performance.

**Important:** REFINE is published prior work, **not our novelty**. The guard rule in this notebook is only an exploratory proof of concept, **not a validated negative-transfer detector**. Do not claim that evaluating two already-observed source conditions demonstrates generalization.

## Step 1 — Settings and GPU

Use the same CUDA-enabled Python/Jupyter environment that worked for your earlier notebook. Install any missing packages with `python -m pip install torch torchvision numpy pandas matplotlib` in your environment (for PyTorch CUDA, use the official selector instead of replacing your working installation).

Run this notebook from the repository root or `notebooks/`. The earlier result directories are resolved from the repository root.

In [ ]:
from pathlib import Path
import random
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import datasets, transforms

seed = 42
batch_size = 16
refine_epochs = 30
pilot_epochs = 5
learning_rate = 0.01

root = Path.cwd().resolve()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'notebooks').is_dir():
    raise RuntimeError('Start Jupyter from the repository root or notebooks directory.')
old40 = root / 'transferguard_noise40_results'
old80 = root / 'transferguard_cifar10_results'
folder = root / 'transferguard_refine_results'
data_dir = root / 'data'
folder.mkdir(parents=True, exist_ok=True)
data_dir.mkdir(parents=True, exist_ok=True)

source_files = {
    40: old40 / 'source_noise40_last.pt',
    80: old80 / 'source_noise80_last.pt'
}

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError('CUDA not available. Select your existing TransferGuard GPU kernel.')
set_seed(seed)
print('GPU:', torch.cuda.get_device_name(0))
print('Current folder:', root)
for noise, path in source_files.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing {noise}% source checkpoint: {path}\nEdit old40 / old80 above.')
    print(f'{noise}% source found:', path)

## Step 2 — Same CIFAR-10 splits as before

These are the **same seed-42 index ranges** used by your earlier 40% and 80% notebooks. We verify any saved split-index files instead of silently replacing them. The 500-image calibration set remains separate from the 2,000-image validation set. We **do not load the official test set** in this notebook.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_data = datasets.CIFAR10(root=str(data_dir), train=True,
                              download=True, transform=transform)
indices = np.random.RandomState(seed).permutation(len(train_data))
source_ids = indices[:10000]
target_ids = indices[10000:14000]
val_ids = indices[14000:16000]
cal_ids = indices[16000:16500]

expected = dict(source=source_ids, target=target_ids,
                validation=val_ids, calibration=cal_ids)
assert len(set(np.concatenate(list(expected.values())))) == 16500

for old_folder in (old40, old80):
    saved_split = old_folder / 'split_indices_seed42.npz'
    if saved_split.exists():
        with np.load(saved_split) as saved:
            assert all(np.array_equal(saved[k], v) for k, v in expected.items()), (
                f'Dataset split mismatch: {saved_split}'
            )
        print('Verified old split:', saved_split)
    else:
        print('Old split file not found (cannot verify):', saved_split)

np.savez(folder / 'split_indices_seed42.npz', **expected)
train_set = Subset(train_data, target_ids)
val_set = Subset(train_data, val_ids)
cal_set = Subset(train_data, cal_ids)

# Keep workers=0 for simple Windows Jupyter behavior.
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                          num_workers=0, pin_memory=True)
val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False,
                        num_workers=0, pin_memory=True)
cal_loader = DataLoader(cal_set, batch_size=batch_size, shuffle=False,
                        num_workers=0, pin_memory=True)
print('Clean target:', len(train_set), 'Validation:', len(val_set),
      'Guard calibration:', len(cal_set))

## Step 3 — Models from the REFINE code

- **BigCNN** is identical in layer sizes and state-dict keys to your previous notebooks. `get_features()` returns its **2,560-dimensional** frozen feature representation.
- **EnhancedCNN (REFINE)** is the paper's three-block target branch (512 dimensions), concatenated with the frozen source representation (2,560 dimensions), followed by a 10-class classifier.

Original architecture: [`model_def_test10/model_def.py`](https://github.com/Xyc-arch/ReFine-residual-feature-integration/blob/main/model_def_test10/model_def.py).

In [ ]:
class BigCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 80, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(80, 160, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(160, 320, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(320, 640, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(640, 640, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(640, 768, 3, padding=1), nn.ReLU()
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(768, 2560), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(2560, 10)
        )

    def get_features(self, x):
        x = self.conv_layers(x).flatten(1)
        return self.fc_layers[1](self.fc_layers[0](x))

    def forward(self, x):
        return self.fc_layers(self.conv_layers(x).flatten(1))


class EnhancedCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 4 * 4, 512), nn.ReLU(), nn.Dropout(0.5)
        )
        self.final_layer = nn.Linear(512 + 2560, 10)

    def forward(self, x, external_features):
        x = self.conv_layers(x).flatten(1)
        features = self.fc_layers(x)
        return self.final_layer(torch.cat([features, external_features], dim=1))


# These checks do not train the model.
with torch.no_grad():
    assert BigCNN().get_features(torch.randn(2, 3, 32, 32)).shape == (2, 2560)
    assert EnhancedCNN()(torch.randn(2, 3, 32, 32), torch.zeros(2, 2560)).shape == (2, 10)
print('Model checks passed.')

## Step 4 — Load source checkpoints and inspect their features

The previous `*_last.pt` files contain a dictionary with a `model` state dict. This loader supports that format. The source CNN is always frozen and in evaluation mode during REFINE training. A collapsed 80%-noise model may have nearly zero feature variation; REFINE's independent target branch is designed to remain trainable in that case.

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

source_hashes = {noise: sha256_file(path) for noise, path in source_files.items()}

def load_source(noise):
    path = source_files[noise]
    saved = torch.load(path, map_location='cpu', weights_only=True)
    weights = saved['model'] if 'model' in saved else saved
    model = BigCNN()
    model.load_state_dict(weights, strict=True)
    model = model.to(device).eval()
    for p in model.parameters():
        p.requires_grad = False
    return model

health_rows = []
for noise in (40, 80):
    source = load_source(noise)
    features = []
    with torch.no_grad():
        for x, _ in cal_loader:
            features.append(source.get_features(x.to(device)).cpu())
    f = torch.cat(features)
    health_rows.append({
        'source_noise': noise,
        'mean_feature_std': f.std(0, unbiased=False).mean().item(),
        'zero_feature_percent': 100 * (f == 0).float().mean().item()
    })
    del source
    torch.cuda.empty_cache()
health = pd.DataFrame(health_rows)
health.to_csv(folder / 'source_health.csv', index=False)
print(health.to_string(index=False))

## Step 5 — REFINE training functions

This follows the paper's `train_enhanced_model`: **SGD, learning rate 0.01, momentum 0.9, cross-entropy**, and frozen external features computed with `no_grad()`.

We add validation tracking and resumable checkpoints (one per epoch). We select the best validation checkpoint without using the official test set. **To save time on your RTX 2050, frozen source features are cached once**: because the source is in evaluation mode and the input transformation has no randomness, this is equivalent to recomputing them every epoch. Only the small EnhancedCNN target branch is trained on the GPU.

**Note:** We use your previous local notebook's fixed 4,000/2,000 target/validation split rather than the paper's exact per-run evaluation protocol. This is a controlled **adaptation**, not a bit-identical reproduction of its reported test accuracies.

In [ ]:
@torch.no_grad()
def evaluate_refine(model, source, loader):
    model.eval()
    source.eval()
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        ext = source.get_features(x)
        correct += (model(x, ext).argmax(1) == y).sum().item()
        total += len(y)
    return 100 * correct / total


class WithFeatures(Dataset):
    def __init__(self, base, features):
        self.base = base
        self.features = features

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        x, y = self.base[i]
        return x, self.features[i], y


@torch.no_grad()
def encode_features(source, base):
    loader = DataLoader(base, batch_size=batch_size, shuffle=False, num_workers=0)
    return torch.cat([source.get_features(x.to(device)).cpu() for x, y in loader])


def get_feature_cache(noise):
    path = folder / f'cached_features_noise{noise}.pt'
    if path.exists():
        saved = torch.load(path, map_location='cpu', weights_only=True)
        assert saved['source_hash'] == source_hashes[noise], 'Cached source weights do not match checkpoint.'
        return saved['train'], saved['validation']

    source = load_source(noise)
    train_features = encode_features(source, train_set)
    val_features = encode_features(source, val_set)
    torch.save({'source_hash': source_hashes[noise],
                'train': train_features, 'validation': val_features}, path)
    del source
    torch.cuda.empty_cache()
    return train_features, val_features


def evaluate_cached(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, ext, y in loader:
            x, ext, y = x.to(device), ext.to(device), y.to(device)
            correct += (model(x, ext).argmax(1) == y).sum().item()
            total += len(y)
    return 100 * correct / total


def train_refine(noise):
    name = f'refine_noise{noise}'
    set_seed(seed)
    train_features, val_features = get_feature_cache(noise)
    train_pairs = DataLoader(WithFeatures(train_set, train_features),
                             batch_size=batch_size, shuffle=True, num_workers=0)
    val_pairs = DataLoader(WithFeatures(val_set, val_features),
                           batch_size=batch_size, shuffle=False, num_workers=0)
    model = EnhancedCNN().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=0.9)
    loss_fn = nn.CrossEntropyLoss()

    last_path = folder / f'{name}_last.pt'
    best_path = folder / f'{name}_best.pt'
    history = []
    start_epoch = 0
    best_acc = -1.0

    if last_path.exists():
        saved = torch.load(last_path, map_location='cpu', weights_only=True)
        assert saved['source_hash'] == source_hashes[noise], 'Source checkpoint changed. Use a new output folder.'
        assert saved['total_epochs'] == refine_epochs, 'Epoch setting changed. Use a new run name.'
        model.load_state_dict(saved['model'])
        optimizer.load_state_dict(saved['optimizer'])
        history = saved['history']
        start_epoch = saved['epoch']
        best_acc = saved['best_acc']
        print('Resuming', name, 'at epoch', start_epoch)

    for epoch in range(start_epoch, refine_epochs):
        # Same order and dropout seed for this epoch even after resume.
        set_seed(seed + epoch)
        model.train()
        total_loss = 0.0

        for x, external, y in train_pairs:
            x, external, y = x.to(device), external.to(device), y.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(x, external), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)

        val_acc = evaluate_cached(model, val_pairs)
        loss_value = total_loss / len(train_set)
        history.append([epoch + 1, loss_value, val_acc])

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), best_path)

        torch.save({
            'epoch': epoch + 1,
            'total_epochs': refine_epochs,
            'source_hash': source_hashes[noise],
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'best_acc': best_acc,
            'history': history
        }, last_path)
        pd.DataFrame(history, columns=['epoch', 'train_loss', 'val_acc']).to_csv(
            folder / f'{name}_history.csv', index=False
        )
        print(f'{name}: {epoch+1:02}/{refine_epochs} | loss {loss_value:.3f} | val {val_acc:.2f}%')

    del model
    torch.cuda.empty_cache()
    return pd.DataFrame(history, columns=['epoch', 'train_loss', 'val_acc'])

## Step 6 — Train REFINE with the 40%-noise source

This trains **only the EnhancedCNN target branch**. The pretrained BigCNN remains frozen.

In [ ]:
refine40 = train_refine(40)
print('40% best validation:', round(refine40.val_acc.max(), 2), '%')

## Step 7 — Train REFINE with the 80%-noise source

Same target architecture, split, optimizer, and schedule. If the source features have collapsed, the trainable target branch can still learn from the clean CIFAR-10 labels.

In [ ]:
refine80 = train_refine(80)
print('80% best validation:', round(refine80.val_acc.max(), 2), '%')

## Step 8 — Compare REFINE against saved baselines

We read **actual CSV files produced by your earlier notebooks**. They are not recreated or hard-coded. Missing histories are shown as missing and the comparison still shows available results.

`best_val_accuracy` is exploratory validation performance, not final test accuracy. The earlier checkpoints may use different run names for the 80% frozen model; the cell handles both.

In [ ]:
old_runs = {
    'scratch_small': [old80 / 'scratch_small_history.csv'],
    'scratch_big': [old40 / 'scratch_big_history.csv', old80 / 'scratch_big_history.csv'],
    'frozen_noise40': [old40 / 'frozen_noise40_reset_history.csv'],
    'frozen_noise80': [old80 / 'frozen_noise80_reset_history.csv',
                       old80 / 'frozen_noise80_history.csv']
}

histories = {}
for name, paths in old_runs.items():
    path = next((p for p in paths if p.exists()), None)
    if path is not None:
        histories[name] = pd.read_csv(path)
    else:
        print('Missing old history:', name, '(put old result folders in the repository root)')

for noise in (40, 80):
    path = folder / f'refine_noise{noise}_history.csv'
    if path.exists():
        histories[f'refine_noise{noise}'] = pd.read_csv(path)

comparison = pd.DataFrame([
    {'method': name, 'best_val_accuracy': h.val_acc.max(),
     'epoch_of_best': int(h.loc[h.val_acc.idxmax(), 'epoch'])}
    for name, h in histories.items()
]).sort_values('best_val_accuracy', ascending=False)
print(comparison.to_string(index=False))
comparison.to_csv(folder / 'validation_comparison.csv', index=False)

plt.figure(figsize=(9, 5))
for name, h in histories.items():
    plt.plot(h.epoch, h.val_acc, label=name)
plt.xlabel('Epoch')
plt.ylabel('Validation accuracy (%)')
plt.title('CIFAR-10: frozen transfer vs REFINE vs scratch')
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(folder / 'validation_curves.png', dpi=150)
plt.show()

## Step 9 — Cheap feature-quality probe (no model training)

A simple nearest-centroid probe asks whether the frozen source's features separate target classes. We build ten class centroids from the **first 500 clean target-training samples**, then evaluate them on the **separate 500-image guard calibration split**. This is a basic probe, *not* LogME or TransRate. It is independent of the official test set.

In [ ]:
probe_train = Subset(train_set, range(500))
probe_loader = DataLoader(probe_train, batch_size=batch_size, shuffle=False,
                          num_workers=0)

@torch.no_grad()
def collect_features(source, loader):
    source.eval()
    fs, ys = [], []
    for x, y in loader:
        fs.append(source.get_features(x.to(device)).cpu())
        ys.append(y.cpu())
    return torch.cat(fs), torch.cat(ys)

probe_rows = []
for noise in (40, 80):
    source = load_source(noise)
    x_train, y_train = collect_features(source, probe_loader)
    x_cal, y_cal = collect_features(source, cal_loader)
    centroids = torch.stack([x_train[y_train == cls].mean(0) for cls in range(10)])
    pred = torch.cdist(x_cal, centroids).argmin(1)
    score = 100 * (pred == y_cal).float().mean().item()
    probe_rows.append({'source_noise': noise, 'centroid_accuracy': score})
    del source
    torch.cuda.empty_cache()

probes = pd.DataFrame(probe_rows)
probes.to_csv(folder / 'feature_probe.csv', index=False)
print(probes.to_string(index=False))

## Step 10 — TransferGuard: exploratory two-signal decision

**Candidate rule, not validated:** use frozen transfer *only if* the source's calibration centroid accuracy is at least 20% **and** its first-five-epoch validation accuracy exceeds scratch SmallCNN by at least 2 percentage points. Otherwise choose scratch. The only two source conditions were already examined when choosing this illustrative rule; **do not report its success on them as independent detector accuracy**.

REFINE is a separate, already-published avoidance baseline; it is **not** a model chosen by this first guard rule. A proper future study needs held-out source conditions, seeds, and guard calibration/validation separation.

In [ ]:
if 'scratch_small' in histories and 'frozen_noise40' in histories and 'frozen_noise80' in histories:
    def pilot_score(history, k=5):
        return float(history.loc[history.epoch <= k, 'val_acc'].iloc[-1])

    scratch_pilot = pilot_score(histories['scratch_small'], pilot_epochs)
    scratch_final = float(histories['scratch_small'].val_acc.max())
    guard_rows = []

    for noise in (40, 80):
        frozen_name = f'frozen_noise{noise}'
        frozen_pilot = pilot_score(histories[frozen_name], pilot_epochs)
        frozen_final = float(histories[frozen_name].val_acc.max())
        centroid_acc = float(probes.loc[probes.source_noise == noise,
                                       'centroid_accuracy'].iloc[0])
        choose_frozen = centroid_acc >= 20 and frozen_pilot >= scratch_pilot + 2
        choice = 'frozen' if choose_frozen else 'scratch'
        chosen_acc = frozen_final if choose_frozen else scratch_final
        guard_rows.append({
            'source_noise': noise,
            'feature_probe': centroid_acc,
            'scratch_pilot': scratch_pilot,
            'frozen_pilot': frozen_pilot,
            'guard_choice': choice,
            'selected_final_val': chosen_acc,
            'oracle_final_val': max(scratch_final, frozen_final)
        })

    guard_results = pd.DataFrame(guard_rows)
    guard_results['regret_points'] = (guard_results.oracle_final_val -
                                      guard_results.selected_final_val)
    guard_results.to_csv(folder / 'guard_demo.csv', index=False)
    print(guard_results.to_string(index=False))
    print('\nExploratory only: these conditions are NOT held-out detector tests.')
else:
    print('Guard demo skipped: historical scratch/frozen CSV files are missing.')
    print('REFINE model training and feature probes are unaffected.')

## Step 11 — Optional held-out test evaluation (only after freezing choices)

The official CIFAR-10 test set has **not** been used above. Run this cell only after deciding which models/checkpoints to report; it evaluates the **best-validation REFINE checkpoints**. It does not tune the guard. Since prior experiments examined test results, treat the overall research program as exploratory unless you create a genuinely untouched final evaluation protocol.

In [ ]:
RUN_FINAL_TEST = False   # Change to True only when the comparison is finalized

if RUN_FINAL_TEST:
    test_data = datasets.CIFAR10(root=str(data_dir), train=False,
                                 download=True, transform=transform)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False,
                             num_workers=0)
    test_rows = []
    for noise in (40, 80):
        source = load_source(noise)
        model = EnhancedCNN().to(device)
        path = folder / f'refine_noise{noise}_best.pt'
        model.load_state_dict(torch.load(path, map_location='cpu', weights_only=True))
        acc = evaluate_refine(model, source, test_loader)
        test_rows.append({'method': f'refine_noise{noise}', 'test_accuracy': acc})
        del source, model
        torch.cuda.empty_cache()
    test_results = pd.DataFrame(test_rows)
    print(test_results.to_string(index=False))
    test_results.to_csv(folder / 'refine_test_results.csv', index=False)
else:
    print('Official test set intentionally not evaluated.')

## What to report to your professor

- **What is reproduced:** REFINE's *model structure and transfer mechanism* (frozen BigCNN 2,560-dim features + 512-dim trainable CNN target branch, concatenation classifier), using **your own saved checkpoints and data split**.
- **What is not reproduced exactly:** the paper's five-run statistics, original source-training trajectories, and the official evaluation protocol.
- **What TransferGuard currently does:** a *preliminary, predefined-for-this-notebook* two-signal frozen-versus-scratch rule. Its first-five-epoch check is based on **historical validation curves**, so it is an exploratory offline replay, not a deployment-ready online selector.
- **What would support a contribution:** a fixed decision rule evaluated on **additional source conditions and paired seeds not used to choose its thresholds**, compared against an early-validation-only selector and reporting compute overhead.

Saved files are in `transferguard_refine_results/`. Run only **Steps 1–8** first if time is short; Steps 9–11 are additional analysis.